# PyTorch-only Colab速度ベンチマーク

このノートブックは、Google Colab上でPyTorchだけを使い、小型seq2seqモデルをゼロから学習してテキスト変換を行う方式の完了時間を計測します。

- 事前学習済み大規模言語モデルやTransformers系ライブラリは使いません。
- Colabは一時環境であり、本番サーバや永続ストレージではありません。
- 結果は別方式と共通のJSON契約で保存します。
- 比較対象はcold startを含む実作業経路の完了時間です。モデル品質の優劣は判定しません。
- 異なるハードウェアで得た秒数は、方式だけでなく実行環境の差も含みます。

## Goal

同じ2入力を3回ずつ処理し、依存import、モデル準備（初期化と学習）、推論、end-to-endの秒数を記録します。勝敗は両方式が成功し、benchmark ID・入力digest・繰り返し回数が一致した場合だけ判定します。

In [ ]:
from pathlib import Path
from time import perf_counter
import hashlib
import json
import platform
import sys

notebook_started_at = perf_counter()

BENCHMARK_ID = "engiiro-transform-speed-v1"
BENCHMARK_CASES = [
    {
        "mode": "baby",
        "input": "今日はチームで仕様書をレビューし、未決事項を整理しました。",
    },
    {
        "mode": "mother",
        "input": "今日はチームで仕様書をレビューし、未決事項を整理しました。",
    },
]
REPEAT_COUNT = 3
TRAINING_EPOCHS = 40
EMBEDDING_SIZE = 48
HIDDEN_SIZE = 96
LEARNING_RATE = 0.01
MAX_OUTPUT_CHARS = 150
RANDOM_SEED = 42

RESULT_PATH = Path("/content/pytorch_colab_benchmark.json")
OTHER_RESULT_PATH = None
DOWNLOAD_RESULT = False

serialized_cases = json.dumps(
    BENCHMARK_CASES,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
)
INPUT_DIGEST = hashlib.sha256(serialized_cases.encode("utf-8")).hexdigest()
print({
    "benchmark_id": BENCHMARK_ID,
    "repeat_count": REPEAT_COUNT,
    "training_epochs": TRAINING_EPOCHS,
    "result_path": str(RESULT_PATH),
})

## Setup

Colabに導入済みのPyTorchをそのまま使います。再インストールは行いません。Python、OS、CPU/GPU、PyTorchの実バージョンを記録します。

In [ ]:
dependency_started_at = perf_counter()
try:
    import torch
    from torch import nn
except ImportError as exc:
    raise RuntimeError(
        "PyTorchがありません。Colabの新しいランタイムを選び直し、再実行してください。"
    ) from exc
dependency_import_seconds = perf_counter() - dependency_started_at

torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

runtime_report = {
    "python": platform.python_version(),
    "os": platform.platform(),
    "device": str(device),
    "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
    "torch": torch.__version__,
}
runtime_report

## Data

秘密情報を含まない小さな固定データだけを使います。速度経路の成立確認用であり、品質評価用ではありません。比較用の2入力は両方式で同一です。

In [ ]:
TRAINING_PAIRS = [
    ("baby", "今日はチームで仕様書をレビューし、未決事項を整理しました。", "きょうは みんなで しようしょをみて、まだきめてないことを まとめたよ。"),
    ("mother", "今日はチームで仕様書をレビューし、未決事項を整理しました。", "今日はチームで仕様書を確認し、未決事項を丁寧に整理しました。"),
    ("baby", "エラーの原因を調査しています。", "えらーさんが どこにいるか しらべてるよ。"),
    ("mother", "エラーの原因を調査しています。", "現在、エラーの原因を落ち着いて確認しています。"),
    ("baby", "明日はデータベースを実装します。", "あしたは でーたべーすを つくるよ。"),
    ("mother", "明日はデータベースを実装します。", "明日はデータベースの実装を進めます。"),
    ("baby", "テストがすべて成功しました。", "てすとが ぜんぶ うまくいったよ。"),
    ("mother", "テストがすべて成功しました。", "すべてのテストが正常に完了しました。"),
]

assert {case["mode"] for case in BENCHMARK_CASES} == {"baby", "mother"}
assert all(mode in {"baby", "mother"} and source and target for mode, source, target in TRAINING_PAIRS)
print({
    "training_pairs": len(TRAINING_PAIRS),
    "benchmark_cases": len(BENCHMARK_CASES),
    "quality_evaluation": False,
})

## Steps

### 1. 文字辞書とTensorを作る

入力はmodeと本文を結合し、出力は文字単位で学習します。処理を単純に保つため、全データを1 batchで扱います。

In [ ]:
PAD_TOKEN = "<pad>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"
SPECIAL_TOKENS = [PAD_TOKEN, BOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

source_texts = [f"{mode}|{source}" for mode, source, _ in TRAINING_PAIRS]
target_texts = [target for _, _, target in TRAINING_PAIRS]
characters = sorted(set("".join(source_texts + target_texts)))
index_to_token = SPECIAL_TOKENS + characters
token_to_index = {token: index for index, token in enumerate(index_to_token)}

PAD_ID = token_to_index[PAD_TOKEN]
BOS_ID = token_to_index[BOS_TOKEN]
EOS_ID = token_to_index[EOS_TOKEN]
UNK_ID = token_to_index[UNK_TOKEN]

def encode_text(text: str) -> list[int]:
    return [BOS_ID, *[token_to_index.get(char, UNK_ID) for char in text], EOS_ID]

def pad_sequences(sequences: list[list[int]]) -> torch.Tensor:
    max_length = max(len(sequence) for sequence in sequences)
    padded = [
        sequence + [PAD_ID] * (max_length - len(sequence))
        for sequence in sequences
    ]
    return torch.tensor(padded, dtype=torch.long, device=device)

source_batch = pad_sequences([encode_text(text) for text in source_texts])
target_batch = pad_sequences([encode_text(text) for text in target_texts])
decoder_input_batch = target_batch[:, :-1]
decoder_label_batch = target_batch[:, 1:]

print({
    "vocabulary_size": len(index_to_token),
    "source_shape": list(source_batch.shape),
    "target_shape": list(target_batch.shape),
})

### 2. 小型seq2seqモデルを定義する

encoderとdecoderはいずれもGRUです。外部モデルや事前学習済み重みは読み込みません。

In [ ]:
model_prepare_started_at = perf_counter()

class TinySeq2Seq(nn.Module):
    def __init__(self, vocabulary_size: int, embedding_size: int, hidden_size: int):
        super().__init__()
        self.embedding = nn.Embedding(
            vocabulary_size,
            embedding_size,
            padding_idx=PAD_ID,
        )
        self.encoder = nn.GRU(embedding_size, hidden_size, batch_first=True)
        self.decoder = nn.GRU(embedding_size, hidden_size, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, vocabulary_size)

    def forward(
        self,
        source_tokens: torch.Tensor,
        decoder_tokens: torch.Tensor,
    ) -> torch.Tensor:
        _, hidden = self.encoder(self.embedding(source_tokens))
        decoded, _ = self.decoder(self.embedding(decoder_tokens), hidden)
        return self.output_layer(decoded)

    def generate(self, source_tokens: torch.Tensor, max_chars: int) -> list[int]:
        _, hidden = self.encoder(self.embedding(source_tokens))
        current = torch.tensor([[BOS_ID]], dtype=torch.long, device=source_tokens.device)
        generated: list[int] = []
        for _ in range(max_chars):
            decoded, hidden = self.decoder(self.embedding(current), hidden)
            next_token = self.output_layer(decoded[:, -1]).argmax(dim=-1)
            token_id = int(next_token.item())
            if token_id == EOS_ID:
                break
            if token_id not in {PAD_ID, BOS_ID, UNK_ID}:
                generated.append(token_id)
            current = next_token.unsqueeze(0)
        return generated

def decode_tokens(token_ids: list[int]) -> str:
    return "".join(index_to_token[token_id] for token_id in token_ids)

model = TinySeq2Seq(
    vocabulary_size=len(index_to_token),
    embedding_size=EMBEDDING_SIZE,
    hidden_size=HIDDEN_SIZE,
).to(device)
print({
    "parameters": sum(parameter.numel() for parameter in model.parameters()),
    "pretrained_weights": False,
})

### 3. 学習する

固定seedで40 epochだけ学習します。lossが有限値で、開始時より低下したことを環境成立の確認とします。

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_history: list[float] = []

model.train()
for _ in range(TRAINING_EPOCHS):
    optimizer.zero_grad(set_to_none=True)
    logits = model(source_batch, decoder_input_batch)
    loss = criterion(
        logits.reshape(-1, logits.shape[-1]),
        decoder_label_batch.reshape(-1),
    )
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss.detach().cpu()))

if device.type == "cuda":
    torch.cuda.synchronize()
model_prepare_seconds = perf_counter() - model_prepare_started_at

assert torch.isfinite(torch.tensor(loss_history)).all()
assert loss_history[-1] < loss_history[0]
print({
    "epochs": TRAINING_EPOCHS,
    "first_loss": round(loss_history[0], 6),
    "final_loss": round(loss_history[-1], 6),
    "model_prepare_seconds": round(model_prepare_seconds, 4),
})

### 4. 同じ2入力を3回ずつ推論する

最初の1回だけ代表出力を保存します。全6回を速度へ含め、生成は最大150文字で停止します。warm-upを別枠にはせず、最初の実行コストも含めます。

In [ ]:
def run_transformation(mode: str, text: str) -> str:
    source = pad_sequences([encode_text(f"{mode}|{text}")])
    with torch.inference_mode():
        token_ids = model.generate(source, max_chars=MAX_OUTPUT_CHARS)
    return decode_tokens(token_ids)

model.eval()
if device.type == "cuda":
    torch.cuda.synchronize()
inference_started_at = perf_counter()

representative_outputs = []
for repeat_index in range(REPEAT_COUNT):
    for case in BENCHMARK_CASES:
        output = run_transformation(case["mode"], case["input"])
        if repeat_index == 0:
            representative_outputs.append({
                "mode": case["mode"],
                "input": case["input"],
                "output": output,
                "output_chars": len(output),
            })

if device.type == "cuda":
    torch.cuda.synchronize()
inference_seconds = perf_counter() - inference_started_at

print({
    "inference_calls": len(BENCHMARK_CASES) * REPEAT_COUNT,
    "inference_seconds": round(inference_seconds, 4),
    "representative_outputs": representative_outputs,
})

## Checks

比較条件、出力件数、文字数、計測値を検査します。出力の自然さや安全性はこの速度ベンチマークでは評価しません。

In [ ]:
assert len(representative_outputs) == len(BENCHMARK_CASES)
assert all(item["output_chars"] <= MAX_OUTPUT_CHARS for item in representative_outputs)
assert model_prepare_seconds > 0
assert inference_seconds > 0
assert INPUT_DIGEST == hashlib.sha256(serialized_cases.encode("utf-8")).hexdigest()

check_report = {
    "condition_contract": "PASS",
    "output_count": "PASS",
    "output_length": "PASS",
    "timing_positive": "PASS",
    "quality_evaluated": False,
}
check_report

## Results

共通JSON契約で結果を保存します。end_to_end_secondsは最初の設定セルから結果作成直前までで、依存import、モデル準備、推論を含みます。Colab起動待ちや手作業時間は含みません。

In [ ]:
benchmark_result = {
    "schema_version": 1,
    "benchmark_id": BENCHMARK_ID,
    "method": "pytorch_colab",
    "success": True,
    "input_digest": INPUT_DIGEST,
    "repeat_count": REPEAT_COUNT,
    "inference_calls": len(BENCHMARK_CASES) * REPEAT_COUNT,
    "environment": runtime_report,
    "timings": {
        "dependency_import_seconds": round(dependency_import_seconds, 4),
        "model_prepare_seconds": round(model_prepare_seconds, 4),
        "inference_seconds": round(inference_seconds, 4),
        "workload_seconds": round(
            dependency_import_seconds + model_prepare_seconds + inference_seconds,
            4,
        ),
        "end_to_end_seconds": round(perf_counter() - notebook_started_at, 4),
    },
    "outputs": representative_outputs,
    "training": {
        "epochs": TRAINING_EPOCHS,
        "first_loss": round(loss_history[0], 6),
        "final_loss": round(loss_history[-1], 6),
        "pretrained_weights": False,
    },
    "quality_evaluated": False,
    "comparison_scope": "cold workflow completion time, not model quality",
}
RESULT_PATH.write_text(
    json.dumps(benchmark_result, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(json.dumps(benchmark_result, ensure_ascii=False, indent=2))

if DOWNLOAD_RESULT:
    from google.colab import files
    files.download(str(RESULT_PATH))

## Optional comparison

もう一方の方式の結果JSONをColabへ置き、OTHER_RESULT_PATHへパスを指定した場合だけ比較します。benchmark ID・入力digest・繰り返し回数が一致し、両方が成功した場合に限り、workload秒が短い方式を表示します。

In [ ]:
def compare_benchmark_results(current: dict, other_path: Path) -> dict:
    other = json.loads(other_path.read_text(encoding="utf-8"))
    comparable_fields = ("benchmark_id", "input_digest", "repeat_count")
    mismatches = [
        field for field in comparable_fields
        if current.get(field) != other.get(field)
    ]
    if mismatches:
        raise ValueError(f"比較条件が一致しません: {', '.join(mismatches)}")
    if not current.get("success") or not other.get("success"):
        raise ValueError("両方式がsuccess=trueの場合だけ時間を比較できます。")

    current_seconds = float(current["timings"]["workload_seconds"])
    other_seconds = float(other["timings"]["workload_seconds"])
    difference = abs(current_seconds - other_seconds)
    winner = "tie"
    if difference > 0.01:
        winner = (
            current["method"]
            if current_seconds < other_seconds
            else other["method"]
        )
    return {
        "winner": winner,
        "difference_seconds": round(difference, 4),
        "basis": "cold workload_seconds",
        "warning": "異なるハードウェアでは、環境と方式を合わせた実作業経路の比較です。",
    }

if OTHER_RESULT_PATH is None:
    print("比較は未実施です。もう一方の結果JSONを置き、OTHER_RESULT_PATHを設定してください。")
else:
    comparison = compare_benchmark_results(
        benchmark_result,
        Path(OTHER_RESULT_PATH),
    )
    print(json.dumps(comparison, ensure_ascii=False, indent=2))

## Next Steps

1. Colabの新しいランタイムで「すべてのセルを実行」します。
2. /content/pytorch_colab_benchmark.json を保存します。
3. 別の通常Python環境で、もう一方の方式を同じ条件で実行します。
4. どちらか一方の比較機能へ相手のJSONを渡します。
5. 勝者はend-to-end時間だけの結果です。出力品質、開発工数、費用、運用性、安全性は別に評価してください。

このノートブックはこのローカル端末では実行していません。Colab上の実測値がない状態で、どちらが速いかは結論づけません。